# Part 2: Time series analysis

Owner: _(name)_

Brief, section 2: use the **training data** to investigate trend, seasonality, stationarity and
autocorrelation with labelled plots and diagnostics; explain how seasonality is detected and report
when evidence is absent; use ADF and/or KPSS with their hypotheses and limitations; justify
differencing or returns; formally define a random walk and its innovation assumptions and discuss
whether the evidence is consistent with it.

Everything below uses the training period only, so that nothing from validation or test leaks into
the analysis.

### Setup (same cell in every notebook)

Runs in Colab and in Cursor. In Colab it clones the repository first; for a private repository
paste a GitHub token when asked (Settings > Developer settings > Fine-grained tokens, read access
to this repository is enough).

In [ ]:
import sys, pathlib

REPO_URL = "https://github.com/Makkaii7/sp500-forecasting.git"   # <- set once after the repo exists

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not pathlib.Path("sp500-forecasting").exists() and not pathlib.Path("src/config.py").exists():
    from getpass import getpass
    token = getpass("GitHub token (leave empty if the repo is public): ")
    url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
    import subprocess
    subprocess.run(["git", "clone", "--quiet", url], check=True)
if IN_COLAB and pathlib.Path("sp500-forecasting").exists():
    import os
    os.chdir("sp500-forecasting")

root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "config.py").exists())
sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src import config, common

df = common.load_prices()
y = common.target_series(df)
train, val, test = common.split(y)
print(root)
print(df.index.min().date(), "to", df.index.max().date(), "|", len(df), "trading days")
print(f"train {train.index.min().date()} to {train.index.max().date()} ({len(train)}) | validation {val.index.min().date()} to {val.index.max().date()} ({len(val)}) | test {test.index.min().date()} to {test.index.max().date()} ({len(test)})")
df.tail(3)

In [ ]:
train_df = df.loc[: config.TRAIN_END]            # analysis on training data only
close = train_df["Close"]
r = train_df["log_return"].dropna()
print("training period:", close.index.min().date(), "to", close.index.max().date(), "|", len(close), "days")

## 1. Trend: the price level against the daily return

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(11, 6))
close.plot(ax=ax[0], title="Training period close: a long-term direction (trend)")
close.rolling(250).mean().plot(ax=ax[0], label="250-day rolling mean"); ax[0].legend(); ax[0].set_ylabel("index points")
r.plot(ax=ax[1], title="Daily log return: wobbles around zero, no trend")
r.rolling(250).mean().plot(ax=ax[1], label="250-day rolling mean"); ax[1].legend()
plt.tight_layout()

## 2. Does the spread change over time? (rolling standard deviation)

In [ ]:
r.rolling(60).std().plot(figsize=(11, 3), title="60-day rolling standard deviation of returns (volatility clusters)")

## 3. Seasonality

A seasonal pattern repeats at a fixed, known period. For daily trading data the candidates are the
week (5 trading days), the month (about 21) and the year (about 252). Three ways to look: calendar
tables (average return by weekday and by month), the ACF at the seasonal lags, and a decomposition.
Report the evidence either way; "no seasonality worth modelling" is a finding.

In [ ]:
by_weekday = r.groupby(r.index.dayofweek).agg(["mean", "std", "count"])
by_weekday.index = ["Mon", "Tue", "Wed", "Thu", "Fri"]
by_month = r.groupby(r.index.month).agg(["mean", "std", "count"])
display(by_weekday.round(5)); display(by_month.round(5))

In [ ]:
from statsmodels.tsa.stattools import acf
lags = [1, 2, 3, 5, 10, 21, 63, 126, 252]
vals = acf(r, nlags=max(lags), fft=True)
limit = 1.96 / np.sqrt(len(r))
pd.DataFrame({"lag": lags, "ACF": [round(vals[k], 4) for k in lags], "beyond ±1.96/√n": [abs(vals[k]) > limit for k in lags]})

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose
dec = seasonal_decompose(close, model="multiplicative", period=252)   # yearly period, as an illustration
fig = dec.plot(); fig.set_size_inches(11, 8); plt.tight_layout()

## 4. Stationarity tests

Two tests with opposite starting assumptions, so read them together:

- **ADF** (augmented Dickey-Fuller). Null hypothesis: the series has a unit root (not stationary).
  A small p-value (under 0.05) is evidence **for** stationarity. Limitation: low power against
  near-unit-root series, and the result depends on the lag choice.
- **KPSS**. Null hypothesis: the series is stationary around a level. A small p-value is evidence
  **against** stationarity. Limitation: p-values are read from a table and are capped at 0.01 and 0.10.

Stationary series: ADF small, KPSS large. Non-stationary series: ADF large, KPSS small.

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss
import warnings

def stationarity_report(series, name):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")          # KPSS warns when the p-value hits the table limits
        adf_stat, adf_p, *_ = adfuller(series.dropna(), autolag="AIC")
        kpss_stat, kpss_p, *_ = kpss(series.dropna(), regression="c", nlags="auto")
    verdict = "stationary" if (adf_p < 0.05 and kpss_p >= 0.05) else "NOT stationary" if (adf_p >= 0.05 and kpss_p < 0.05) else "mixed, look at the plots"
    print(f"{name:12s} ADF stat = {adf_stat:7.3f} p = {adf_p:.4f}   KPSS stat = {kpss_stat:6.3f} p = {kpss_p:.4f}   -> {verdict}")

stationarity_report(close, "Close")
stationarity_report(r, "log return")

## 5. ACF and PACF: how far back does the past matter?

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
fig, ax = plt.subplots(2, 2, figsize=(12, 7))
plot_acf(close, lags=40, ax=ax[0, 0], title="ACF of Close (decays slowly: not stationary)")
plot_pacf(close, lags=40, ax=ax[0, 1], title="PACF of Close")
plot_acf(r, lags=40, ax=ax[1, 0], title="ACF of log return (inside the limits: close to white noise)")
plot_pacf(r, lags=40, ax=ax[1, 1], title="PACF of log return")
plt.tight_layout()

## 6. The random walk, formally

A random walk says tomorrow's value is today's value plus a surprise:

$$y_t = y_{t-1} + \varepsilon_t \qquad (\text{with drift: } y_t = c + y_{t-1} + \varepsilon_t)$$

The surprises, called innovations, are assumed to be white noise: mean zero, constant variance,
and uncorrelated with each other and with the past. This is ARIMA(0, 1, 0) in the lecture's
notation. Under it, the best forecast of tomorrow's close is today's close (plus the drift), which is
exactly the baseline every model in this report must beat.

Evidence to check: the price level fails the stationarity tests (unit root), the first difference
(the return) passes them, and the return's autocorrelations are jointly zero. The Ljung-Box test
below checks the last point: its null hypothesis is that the first k autocorrelations are all zero.

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox
lb = acorr_ljungbox(r, lags=[5, 10, 20], return_df=True)
lb["verdict"] = np.where(lb["lb_pvalue"] < 0.05, "some autocorrelation", "consistent with white noise")
lb

## 7. Decision

Write the verdict in plain words, for example:

- The price level is not stationary (ADF p large, KPSS p small, ACF decays slowly). One difference
  of the log price, the daily log return, is stationary. So d = 1 on the price, equivalently d = 0 on the return.
- Seasonality: (what the calendar tables, seasonal-lag ACF and decomposition showed, and whether anything is worth modelling).
- Random walk: (whether the evidence is consistent with it: innovations near white noise or not, and any
  autocorrelation found at low lags).
- Split dates for `src/config.py`: TRAIN_END = ..., VAL_END = ... (reason: ...).